In [1]:
import numpy as np
from numba import cuda
import math

@cuda.jit
def bitonic_sort_kernel(d_array, j, k):
    """
    Core Bitonic Sort Kernel.
    Each thread handles a comparison and potential swap between two elements.
    """
    i = cuda.grid(1)

    # Target index to compare with
    ixj = i ^ j

    # Ensure each pair is only compared once by checking the relationship
    if ixj > i:
        if (i & k) == 0:
            # Sort ascending
            if d_array[i] > d_array[ixj]:
                d_array[i], d_array[ixj] = d_array[ixj], d_array[i]
        else:
            # Sort descending
            if d_array[i] < d_array[ixj]:
                d_array[i], d_array[ixj] = d_array[ixj], d_array[i]

def bitonic_sort(h_array):
    n = h_array.size

    if (n & (n - 1)) != 0 or n == 0:
        raise ValueError("Array size must be a perfect power of 2 for Bitonic Sort.")

    # Allocate device memory and copy data
    d_array = cuda.to_device(h_array.astype(np.int32))

    # Configure execution grid layout
    threads_per_block = 256
    blocks_per_grid = math.ceil(n / threads_per_block)

    # Outer loop dictates the size of the bitonic sequence (powers of 2)
    k = 2
    while k <= n:
        # Inner loop dictates the stride/stage interval of comparison
        j = k >> 1
        while j > 0:
            # Launch kernel for parallel comparisons
            bitonic_sort_kernel[blocks_per_grid, threads_per_block](d_array, j, k)
            cuda.synchronize() # Wait for all swaps to finish before the next step
            j >>= 1
        k <<= 1

    # Copy sorted data back to the host CPU
    return d_array.copy_to_host()


    array_size = 1024

    # Generate a random unsorted array of integers
    np.random.seed(42)
    test_array = np.random.randint(0, 5000, size=array_size).astype(np.int32)

    print("Original Array (First 15 elements):")
    print(test_array[:15])
    print("-" * 50)

    # Run GPU Bitonic Sort
    gpu_sorted = bitonic_sort(test_array)

    # Run CPU baseline sort for verification
    cpu_sorted = np.sort(test_array)

    print("GPU Sorted Array (First 15 elements):")
    print(gpu_sorted[:15])
    print("-" * 50)

    # Verify correctness
    if np.array_equal(gpu_sorted, cpu_sorted):
        print("Success! The GPU Bitonic Sort completely matches the CPU sort.")
    else:
        print("Mismatch! The sorting was incorrect.")

In [2]:
# THIS IS THE BLOCK THAT ACTUALLY TRIGGERS THE PRINT STATEMENTS
if __name__ == "__main__":
    array_size = 1024
    np.random.seed(42)
    test_array = np.random.randint(0, 5000, size=array_size).astype(np.int32)

    # These are what generate your output window text:
    print("Original Array (First 15 elements):")
    print(test_array[:15])
    print("-" * 50)

    gpu_sorted = bitonic_sort(test_array)

    print("GPU Sorted Array (First 15 elements):")
    print(gpu_sorted[:15])

Original Array (First 15 elements):
[ 860 3772 3092  466 4426 3444 3171 2919  130 1685  769 2391 2433 1184
 4555]
--------------------------------------------------


/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:696: NumbaPerformanceWarning: Grid size 4 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


GPU Sorted Array (First 15 elements):
[  3   4   5   9   9  16  34  55  64  66  77  94  98 114 116]
